In [12]:
import pandas as pd
import numpy as np

In [13]:
regions = [
    "Nairobi","Mombasa","Kisumu","Nakuru","Kiambu","Machakos","Kajiado","Uasin Gishu",
    "Kakamega","Bungoma","Busia","Siaya","Homa Bay","Migori","Kisii","Nyamira",
    "Kericho","Bomet","Narok","Turkana","West Pokot","Samburu","Trans Nzoia",
    "Elgeyo Marakwet","Nandi","Laikipia","Nyandarua","Murang'a","Kirinyaga",
    "Nyeri","Embu","Tharaka Nithi","Meru","Isiolo","Marsabit","Garissa",
    "Wajir","Mandera","Tana River","Lamu","Taita Taveta","Kwale","Kilifi",
    "Kitui","Makueni","Vihiga"
]
years = list(range(2010, 2025))
diseases = ["Malaria", "Cholera", "Tuberculosis"]

In [15]:
data = []

for region in regions:
    for year in years:
        for disease in diseases:
            data.append([region, year, disease])

df = pd.DataFrame(data, columns=["region", "year", "disease"])

df.tail()

,region,year,disease
2065,Vihiga,2023,Cholera
2066,Vihiga,2023,Tuberculosis
2067,Vihiga,2024,Malaria
2068,Vihiga,2024,Cholera
2069,Vihiga,2024,Tuberculosis


In [16]:
np.random.seed(42)

df["population"] = np.random.randint(500000, 5000000, size=len(df))

In [17]:
df["rainfall"] = np.random.uniform(500, 2000, size=len(df))
df["temperature"] = np.random.uniform(15, 35, size=len(df))

In [18]:
def generate_cases(row):
    base = 0

    if row["disease"] == "Malaria":
        base = row["rainfall"] * 0.8 + row["temperature"] * 10
    elif row["disease"] == "Cholera":
        base = row["rainfall"] * 0.5
    elif row["disease"] == "Tuberculosis":
        base = row["population"] * 0.0002

    noise = np.random.randint(0, 500)
    return int(base + noise)

df["cases"] = df.apply(generate_cases, axis=1)

In [19]:
def classify_risk(row):
    rate = (row["cases"] / row["population"]) * 1000

    if rate < 1:
        return "Low"
    elif rate < 3:
        return "Medium"
    else:
        return "High"

df["risk_level"] = df.apply(classify_risk, axis=1)

In [20]:
df.to_csv("health_dataset.csv", index=False)

In [21]:
import pandas as pd

df = pd.read_csv("health_dataset.csv")

df.head()

,region,year,disease,population,rainfall,temperature,cases,risk_level
0,Nairobi,2010,Malaria,2192743,1809.984374,31.698944,2128,Low
1,Nairobi,2010,Cholera,4804572,1454.437031,34.493812,1216,Low
2,Nairobi,2010,Tuberculosis,2734489,1641.682306,17.709191,584,Low
3,Nairobi,2011,Malaria,2070006,740.107452,19.629504,876,Low
4,Nairobi,2011,Cholera,1636074,1192.336212,32.368190,1025,Low


In [22]:
from sklearn.preprocessing import LabelEncoder

le_region = LabelEncoder()
le_disease = LabelEncoder()

df["region_encoded"] = le_region.fit_transform(df["region"])
df["disease_encoded"] = le_disease.fit_transform(df["disease"])

In [23]:
X = df[[
    "region_encoded",
    "year",
    "disease_encoded",
    "population",
    "rainfall",
    "temperature"
]]

y = df["cases"]

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
from sklearn.ensemble import RandomForestRegressor

model_cases = RandomForestRegressor()

model_cases.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
predictions = model_cases.predict(X_test)

from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, predictions)

print("MAE:", mae)

MAE: 128.72545893719806


In [27]:
import joblib

joblib.dump(model_cases, "cases_model.pkl")
joblib.dump(le_region, "le_region.pkl")
joblib.dump(le_disease, "le_disease.pkl")

['le_disease.pkl']

In [28]:
sample = X_test.iloc[0:1]

real = y_test.iloc[0]
pred = model_cases.predict(sample)[0]

print("Real cases:", real)
print("Predicted cases:", pred)

Real cases: 524
Predicted cases: 460.08


In [29]:
df["risk_level"]

0       Low
1       Low
2       Low
3       Low
4       Low
       ... 
2065    Low
2066    Low
2067    Low
2068    Low
2069    Low
Name: risk_level, Length: 2070, dtype: object

In [30]:
le_risk = LabelEncoder()

df["risk_encoded"] = le_risk.fit_transform(df["risk_level"])

In [31]:
X = df[[
    "region_encoded",
    "year",
    "disease_encoded",
    "population",
    "rainfall",
    "temperature",
    "cases"
]]

y = df["risk_encoded"]

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
from sklearn.ensemble import RandomForestClassifier

model_risk = RandomForestClassifier()

model_risk.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
from sklearn.metrics import accuracy_score

predictions = model_risk.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.9710144927536232


In [35]:
import joblib

joblib.dump(model_risk, "risk_model.pkl")
joblib.dump(le_risk, "le_risk.pkl")

['le_risk.pkl']

In [36]:
sample = X_test.iloc[0:1]

# Predict risk
pred_risk = model_risk.predict(sample)[0]

# Convert back to label
risk_label = le_risk.inverse_transform([pred_risk])[0]

print("Predicted Risk:", risk_label)

Predicted Risk: Low
